# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step workflow for loading, exploring, and processing the FAIR^2 dataset (clinicopathological data for 77 cancer survivors who developed a second primary colorectal cancer) using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset schema is provided as a Croissant JSON-LD at the following URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load the metadata and data records from the dataset using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Instantiate the Dataset object
dataset = mlc.Dataset(croissant_url)

# Access and print dataset metadata as an object
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")
print(f"Identifier: {dataset.metadata.identifier}")

## 2. Data Overview
Let's examine the available record sets, their `@id`s, and fields. This allows us to decide which parts of the data to analyze further.

In [ ]:
# List all available record sets in the dataset by @id and display their field @ids
record_sets = dataset.record_sets()
print("Available Record Sets and their fields:")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  RecordSet @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id})  | dataType: {field.data_type}")
    print("")

## 3. Data Extraction
We'll load the main patient clinical data record set (the one that contains tabular clinical and pathological variables). All references use the unique `@id` string from the dataset specification.

In [ ]:
# Choose the main clinical record set by inspecting previous output.
# We'll programmatically grab the first record set for demonstration, you can adjust if needed.

# Collect all record set ids
record_set_ids = [rs.id for rs in dataset.record_sets()]
# Display all record set ids
print("RecordSet @ids:")
for id_ in record_set_ids:
    print(f"  - {id_}")

# If documentation describes a particular record set for main patient clinical variables, select it here.
# For now, we'll use the first record set as main example.
main_record_set_id = record_set_ids[0]

# Load records from each record set into pandas DataFrames
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for RecordSet: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

print(f"\nColumns in main record set ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
print("\nPreview of data:")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We now process the extracted data. We'll:
- Filter records based on a numeric variable (such as age or time interval)
- Normalize a numeric column
- Group by a relevant categorical field (for example, MSI status or anatomical location)

All field accesses and grouping are performed using each field's `@id`. Adjust field names/IDs as appropriate for your analytic question.

In [ ]:
# Identify a numeric field (e.g. patient age, if available, by its @id)
# For this demonstration, we'll list columns and pick an evidently numeric candidate
main_df = dataframes[main_record_set_id]

print("All columns in the main DataFrame:")
for col in main_df.columns:
    print(f"- {col}")

# Suppose 'http://senscience.ai/age_at_second_primary_diagnosis' is a field representing age (replace with actual @id as appropriate)
# If unsure, after printing columns above, select a likely numeric column @id below
numeric_field_id = None
for col in main_df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
        break
# Fallback if no such field
if numeric_field_id is None:
    numeric_field_id = main_df.select_dtypes(include=['number']).columns[0]

print(f"\nUsing as numeric field: {numeric_field_id}")

# Filter for patients with age > 60 (or for another meaningful threshold if your numeric field differs)
threshold = 60
filtered_df = main_df.copy()
filtered_df = filtered_df[pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') > threshold]

print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()

print("\nNormalized values of the numeric field:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Pick a group-by field: look for an anatomical location or MSI status field
group_field_id = None
for col in main_df.columns:
    if 'msi' in col.lower() or 'status' in col.lower() or 'site' in col.lower():
        group_field_id = col
        break
# If nothing suitable, just use the first object-type column
if group_field_id is None:
    group_field_id = main_df.select_dtypes(include=['object']).columns[0]

print(f"\nGrouping by field: {group_field_id}")

# Group and show means of numeric fields by group_field_id
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Let's visualize key data distributions or relationships. Below is an example plotting the distribution of the selected numeric variable, colored by the chosen group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Boxplot or violin plot for the numeric field across group levels
plt.figure(figsize=(8,5))
if group_field_id in filtered_df.columns:
    sns.boxplot(
        x=group_field_id, 
        y=numeric_field_id, 
        data=filtered_df
    )
    plt.ylabel(numeric_field_id)
    plt.xlabel(group_field_id)
    plt.title(f"{numeric_field_id} distribution by {group_field_id}")
    plt.xticks(rotation=45)
else:
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
plt.tight_layout()
plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded structured clinical data on second primary colorectal cancer survivors using the Croissant schema and the `mlcroissant` Python API
- Identified and selected fields using their unique `@id`s for clean programmatic access
- Filtered, normalized, and grouped data for initial statistical analysis
- Visualized relationships to uncover potential patterns in age and clinical/pathological factors

Explore further by adjusting the selected record set and field `@id`s to match your analytical hypotheses! All operations are transparent and reproducible using the Croissant dataset structure.